# Reddit — Data Preparation

## What this notebook does

The raw Reddit data was collected via the Reddit API and stored as **`.jsonl` files** (JSON Lines — one JSON record per line) in the Bronze folder, one file per subreddit. The analysis notebooks expect two **combined `.parquet` files**:

- `reddit_comments_raw.parquet` — all comments across all subreddits
- `reddit_posts_raw.parquet` — all posts across all subreddits

This notebook merges and converts them.

**Why parquet?** Parquet is a compressed columnar format. It is ~5–10× faster to load than JSON for large dataframes and uses significantly less disk space.

**Input:** `Data/1_Bronze/Reddit/r_*_comments.jsonl`, `r_*_posts.jsonl`  
**Output:** `Data/1_Bronze/Reddit/reddit_comments_raw.parquet`, `reddit_posts_raw.parquet`

## 0. Setup

In [1]:
import json
import pandas as pd
from pathlib import Path

BRONZE = Path('../../Data/1_Bronze/Reddit')
print('Setup complete.')
print(f'Bronze folder: {BRONZE.resolve()}')

Setup complete.
Bronze folder: C:\Users\ninav\OneDrive - UGent\Unif\2025-2026\Semester 2\Social Media and Web Analytics\group-project-SMWA\Data\1_Bronze\Reddit


## 1. Inspect available files

In [2]:
jsonl_files = sorted(BRONZE.glob('*.jsonl'))
print(f'Found {len(jsonl_files)} .jsonl files:')
for f in jsonl_files:
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<45}  {size_mb:.1f} MB')

Found 14 .jsonl files:
  r_conservative_comments.jsonl                  1136.2 MB
  r_conservative_posts.jsonl                     157.9 MB
  r_democrats_comments.jsonl                     767.2 MB
  r_democrats_posts.jsonl                        88.2 MB
  r_liberal_comments.jsonl                       33.4 MB
  r_liberal_posts.jsonl                          6.3 MB
  r_politics_comments.jsonl                      3783.9 MB
  r_politics_posts.jsonl                         288.0 MB
  r_republican_comments.jsonl                    166.1 MB
  r_republican_posts.jsonl                       36.7 MB
  r_trump_comments.jsonl                         320.2 MB
  r_trump_posts.jsonl                            80.7 MB
  r_worldnews_comments.jsonl                     3228.1 MB
  r_worldnews_posts.jsonl                        147.8 MB


## 2. Convert and save to parquet incrementally

Instead of loading all files into memory at once, we process one file at a time and
append each batch directly to the parquet file using `pyarrow.ParquetWriter`.
This keeps memory usage low and avoids needing temporary disk space for large intermediate DataFrames.

In [3]:
import pyarrow as pa
import pyarrow.parquet as pq

# Output folders — one parquet file per subreddit inside each folder
comments_dir = BRONZE / 'comments'
posts_dir    = BRONZE / 'posts'
comments_dir.mkdir(exist_ok=True)
posts_dir.mkdir(exist_ok=True)

CHUNKSIZE = 50_000

# Numeric promotion order: higher rank wins when types conflict
_TYPE_RANK = {pa.bool_(): 0, pa.int32(): 1, pa.int64(): 2,
              pa.float32(): 3, pa.float64(): 4, pa.large_utf8(): 99}

def _wider(t1, t2):
    if t1 == t2:
        return t1
    r1 = _TYPE_RANK.get(t1, -1)
    r2 = _TYPE_RANK.get(t2, -1)
    if r1 >= 0 and r2 >= 0:
        return t1 if r1 >= r2 else t2
    return pa.large_utf8()  # fallback: convert to string

def merge_schemas(s1, s2):
    """Merge two Arrow schemas field by field, promoting conflicting types."""
    fields = {f.name: f.type for f in s1}
    for f in s2:
        fields[f.name] = _wider(fields[f.name], f.type) if f.name in fields else f.type
    return pa.schema([pa.field(n, t) for n, t in fields.items()])

def drop_nested_columns(df):
    """Drop columns containing dicts or lists — not parquet-compatible."""
    nested = [c for c in df.columns
              if df[c].dropna().apply(lambda x: isinstance(x, (dict, list))).any()]
    return df.drop(columns=nested)

def iter_chunks(f):
    """Yield cleaned pandas chunks from a JSONL file."""
    for chunk in pd.read_json(f, lines=True, chunksize=CHUNKSIZE):
        chunk['subreddit'] = chunk['subreddit'].str.lower()
        if 'created_utc' in chunk.columns:
            chunk['created_utc'] = pd.to_datetime(chunk['created_utc'], unit='s', utc=True)
        yield drop_nested_columns(chunk)

def build_unified_schema(f):
    """Pass 1: scan all chunks to build a unified Arrow schema."""
    schema = None
    for chunk in iter_chunks(f):
        tbl = pa.Table.from_pandas(chunk, preserve_index=False)
        schema = tbl.schema if schema is None else merge_schemas(schema, tbl.schema)
    return schema

def align_table(table, schema):
    """Align a table to the target schema: reorder cols, add missing as null, cast types."""
    arrays = []
    for field in schema:
        if field.name in table.schema.names:
            col = table.column(field.name)
            if col.type != field.type:
                try:
                    col = col.cast(field.type, safe=False)
                except Exception:
                    col = pa.nulls(len(table), type=field.type)
            arrays.append(col)
        else:
            arrays.append(pa.nulls(len(table), type=field.type))
    return pa.table(dict(zip(schema.names, arrays)), schema=schema)

def convert_file(f, out_dir):
    out_path = out_dir / f.name.replace('.jsonl', '.parquet')

    print(f'  {f.name} — building schema (pass 1/2)...', end='\r')
    schema = build_unified_schema(f)

    writer = pq.ParquetWriter(out_path, schema)
    total = 0
    for chunk in iter_chunks(f):
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        table = align_table(table, schema)
        writer.write_table(table)
        total += len(chunk)
    writer.close()
    print(f'  {f.name:<45}  {total:>7,} records  →  {out_path.name}')

print('Converting comments...')
for f in sorted(BRONZE.glob('*_comments.jsonl')):
    convert_file(f, comments_dir)

print('\nConverting posts...')
for f in sorted(BRONZE.glob('*_posts.jsonl')):
    convert_file(f, posts_dir)

print('\nDone.')
print(f'Comments folder: {comments_dir}')
print(f'Posts folder   : {posts_dir}')

Converting comments...
  r_conservative_comments.jsonl                  693,340 records  →  r_conservative_comments.parquet
  r_democrats_comments.jsonl                     444,280 records  →  r_democrats_comments.parquet
  r_liberal_comments.jsonl                        19,652 records  →  r_liberal_comments.parquet
  r_politics_comments.jsonl                      2,104,000 records  →  r_politics_comments.parquet
  r_republican_comments.jsonl                     93,495 records  →  r_republican_comments.parquet
  r_trump_comments.jsonl                         188,947 records  →  r_trump_comments.parquet
  r_worldnews_comments.jsonl                     1,847,724 records  →  r_worldnews_comments.parquet

Converting posts...
  r_conservative_posts.jsonl                      41,790 records  →  r_conservative_posts.parquet
  r_democrats_posts.jsonl                         20,756 records  →  r_democrats_posts.parquet
  r_liberal_posts.jsonl                            1,887 records  →  r_liber

## 3. Merge per-subreddit parquet files into one combined file

Combines all individual subreddit parquet files into:
- `comments/reddit_comments_raw.parquet`
- `posts/reddit_posts_raw.parquet`

Uses streaming row-group iteration so memory usage stays low even for large datasets.

In [4]:
def merge_parquet_dir(input_dir, out_path):
    """Merge all parquet files in input_dir into a single parquet file at out_path."""
    files = sorted(input_dir.glob('*.parquet'))
    if not files:
        print(f'  No parquet files found in {input_dir}')
        return

    # Pass 1: build unified schema across all files
    schema = None
    for f in files:
        s = pq.read_schema(f)
        # Strip pandas metadata to avoid conflicts; keep only field names/types
        s = pa.schema([pa.field(f.name, f.type) for f in s])
        schema = s if schema is None else merge_schemas(schema, s)

    # Pass 2: stream row groups from each file and write to combined file
    writer = pq.ParquetWriter(out_path, schema)
    total = 0
    for f in files:
        pf = pq.ParquetFile(f)
        n = 0
        for batch in pf.iter_batches(batch_size=100_000):
            table = pa.Table.from_batches([batch])
            table = align_table(table, schema)
            writer.write_table(table)
            n += len(table)
        total += n
        print(f'  {f.name:<45}  {n:>7,} records')
    writer.close()
    print(f'  → {out_path.name}  ({total:,} records total)\n')


print('Merging comments...')
merge_parquet_dir(comments_dir, comments_dir / 'reddit_comments_raw.parquet')

print('Merging posts...')
merge_parquet_dir(posts_dir, posts_dir / 'reddit_posts_raw.parquet')

print('Done.')
print(f'  {comments_dir / "reddit_comments_raw.parquet"}')
print(f'  {posts_dir / "reddit_posts_raw.parquet"}')

Merging comments...
  r_conservative_comments.parquet                693,340 records
  r_democrats_comments.parquet                   444,280 records
  r_liberal_comments.parquet                      19,652 records
  r_politics_comments.parquet                    2,104,000 records
  r_republican_comments.parquet                   93,495 records
  r_trump_comments.parquet                       188,947 records
  r_worldnews_comments.parquet                   1,847,724 records
  → reddit_comments_raw.parquet  (5,391,438 records total)

Merging posts...
  r_conservative_posts.parquet                    41,790 records
  r_democrats_posts.parquet                       20,756 records
  r_liberal_posts.parquet                          1,887 records
  r_politics_posts.parquet                        69,963 records
  r_republican_posts.parquet                       9,707 records
  r_trump_posts.parquet                           19,111 records
  r_worldnews_posts.parquet                       36,8